### Assignment 2 - Solar Power Forecasting with a Graph Neural Network
- Discipline: EEE6553 - Advanced Deep Learning
- Task: time-ordered solar power forecasting using a custom message-passing GNN

### 1. Libraries and Environment Setup
Import the required libraries and verify whether TensorFlow can use a GPU in the current notebook kernel.

In [3]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf

try:
    from tensorflow.keras import layers, Model
    _keras = tf.keras
except (ImportError, ModuleNotFoundError):
    import keras
    from keras import layers, Model
    _keras = keras

tf_version = tf.__version__
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf_version)

gpus = tf.config.list_physical_devices("GPU")
print("GPUs detected:", gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU is available. Training can use GPU.")
else:
    print("No GPU detected. Training will run on CPU.")
    print("For Windows GPU: switch kernel to 'Python (tf210-gpu)' - Python 3.10 + TF 2.10.")

Python: 3.11.9
TensorFlow: 2.21.0
GPUs detected: []
No GPU detected. Training will run on CPU.
For Windows GPU: switch kernel to 'Python (tf210-gpu)' - Python 3.10 + TF 2.10.


## 2. Dataset Loading and Problem Definition
Load the solar forecasting dataset, inspect its structure, and define which columns represent time, input features, and the prediction target.

In [4]:
candidate_files = [
    "/content/solar power forecasting.csv",
    "solar power forecasting.csv",
    "datasets/solar power forecasting.csv",
    "./datasets/solar power forecasting.csv",
]
file_path = next((path for path in candidate_files if os.path.isfile(path)), None)
assert file_path is not None, "Dataset file not found. Checked: " + ", ".join(candidate_files)
print("Using dataset file:", file_path)

df = pd.read_csv(file_path)
print("Shape of dataset:", df.shape)
print(df.head())

time_col = df.columns[0]
feature_cols = df.columns[1:-1]
target_col = df.columns[-1]

print("Time column   :", time_col)
print("Feature cols  :", list(feature_cols))
print("Target column :", target_col)

Using dataset file: solar power forecasting.csv
Shape of dataset: (791, 11)
                  time    GHI    DNI    DHI  Temperature  Relative Humidity  \
0  2018-07-01 06:00:00  143.0  436.0   56.0         16.2              84.87   
1  2018-07-01 06:30:00  234.0  537.0   76.0         16.9              81.27   
2  2018-07-01 07:00:00  333.0  624.0   90.0         17.6              77.79   
3  2018-07-01 07:30:00  436.0  707.0   96.0         18.4              73.98   
4  2018-07-01 08:00:00  535.0  757.0  105.0         19.2              69.10   

   Pressure  Wind Speed  Solar Zenith Angle  Cloud Type        pv  
0    1014.0         3.3               78.51         0.0  0.077255  
1    1015.0         3.2               72.89         0.0  0.410263  
2    1015.0         3.1               67.13         0.0  1.425640  
3    1015.0         2.9               61.28         0.0  3.315729  
4    1015.0         2.7               55.37         0.0  5.809145  
Time column   : time
Feature cols  : ['GH

## 3. Feature Scaling and Temporal Graph Construction
Standardize the feature and target values, then build a simple temporal adjacency matrix that connects each time step to its neighbors.

In [5]:
X = df[feature_cols].values.astype(np.float32)
y = df[target_col].values.astype(np.float32)

x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X).astype(np.float32)

y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).astype(np.float32).flatten()

num_nodes = X_scaled.shape[0]
num_features = X_scaled.shape[1]
print("Number of nodes   :", num_nodes)
print("Number of features:", num_features)

A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
for i in range(num_nodes):
    A[i, i] = 1.0
    if i > 0:
        A[i, i - 1] = 1.0
    if i < num_nodes - 1:
        A[i, i + 1] = 1.0

row_sum = A.sum(axis=1, keepdims=True)
A = A / row_sum
print("Adjacency shape:", A.shape)

Number of nodes   : 791
Number of features: 9
Adjacency shape: (791, 791)


## 4. Train/Test Split and Tensor Preparation
Preserve time order during the split, create training and test masks, and convert the data structures into TensorFlow tensors with a batch dimension.

In [6]:
split_index = int(0.8 * num_nodes)

train_mask = np.zeros(num_nodes, dtype=np.float32)
test_mask = np.zeros(num_nodes, dtype=np.float32)
train_mask[:split_index] = 1.0
test_mask[split_index:] = 1.0

X_tensor = tf.convert_to_tensor(X_scaled, dtype=tf.float32)
A_tensor = tf.convert_to_tensor(A, dtype=tf.float32)
y_tensor = tf.convert_to_tensor(y_scaled.reshape(-1, 1), dtype=tf.float32)
train_mask_tensor = tf.convert_to_tensor(train_mask.reshape(-1, 1), dtype=tf.float32)
test_mask_tensor = tf.convert_to_tensor(test_mask.reshape(-1, 1), dtype=tf.float32)

X_tensor = tf.expand_dims(X_tensor, axis=0)
A_tensor = tf.expand_dims(A_tensor, axis=0)
y_tensor = tf.expand_dims(y_tensor, axis=0)
train_mask_tensor = tf.expand_dims(train_mask_tensor, axis=0)
test_mask_tensor = tf.expand_dims(test_mask_tensor, axis=0)

## 5. Custom GNN Architecture
Define the custom message-passing layer and the regression model that applies multiple residual graph-update steps before producing a forecast for each node.

In [7]:
class MessagePassingLayer(layers.Layer):
    def __init__(self, hidden_units, dropout_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.message_mlp = _keras.Sequential([
            layers.Dense(hidden_units, activation='relu'),
            layers.Dense(hidden_units, activation='relu')
        ])
        self.update_mlp = _keras.Sequential([
            layers.Dense(hidden_units, activation='relu'),
            layers.Dropout(dropout_rate),
            layers.Dense(hidden_units, activation='relu')
        ])

    def call(self, inputs, training=False):
        X, A = inputs
        messages = self.message_mlp(X, training=training)
        aggregated = tf.matmul(A, messages)
        combined = tf.concat([X, aggregated], axis=-1)
        return self.update_mlp(combined, training=training)

class GNNRegressor(Model):
    def __init__(self, hidden_units=32, num_gnn_layers=3, dropout_rate=0.2):
        super().__init__()
        self.input_proj = layers.Dense(hidden_units, activation='relu')
        self.gnn_layers = [
            MessagePassingLayer(hidden_units=hidden_units, dropout_rate=dropout_rate)
            for _ in range(num_gnn_layers)
        ]
        self.regressor = _keras.Sequential([
            layers.Dense(hidden_units, activation='relu'),
            layers.Dense(hidden_units // 2, activation='relu'),
            layers.Dense(1)
        ])

    def call(self, inputs, training=False):
        X, A = inputs
        H = self.input_proj(X)
        for gnn_layer in self.gnn_layers:
            H = H + gnn_layer([H, A], training=training)
        return self.regressor(H, training=training)

## 6. Optimization, Training, and Final Evaluation
Initialize the optimizer, train the GNN on the training mask, and report the final regression metrics on the held-out test period.

In [8]:
model = GNNRegressor(hidden_units=64, num_gnn_layers=3, dropout_rate=0.2)
optimizer = _keras.optimizers.Adam(learning_rate=0.001)
epochs = 100

def masked_mse(y_true, y_pred, mask):
    error = tf.square(y_true - y_pred)
    error = error * mask
    return tf.reduce_sum(error) / (tf.reduce_sum(mask) + 1e-8)

for epoch in range(epochs):
    with tf.GradientTape() as tape:
        y_pred = model([X_tensor, A_tensor], training=True)
        loss = masked_mse(y_tensor, y_pred, train_mask_tensor)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if (epoch + 1) % 20 == 0:
        test_pred = model([X_tensor, A_tensor], training=False)
        test_loss = masked_mse(y_tensor, test_pred, test_mask_tensor)
        print(f"Epoch {epoch + 1:03d} | Train Loss: {loss.numpy():.6f} | Test Loss: {test_loss.numpy():.6f}")

pred_scaled = model([X_tensor, A_tensor], training=False).numpy()[0, :, 0]
pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).flatten()
y_true = y

pred_test = pred[split_index:]
y_test_real = y_true[split_index:]

rmse = np.sqrt(mean_squared_error(y_test_real, pred_test))
mae = mean_absolute_error(y_test_real, pred_test)
r2 = r2_score(y_test_real, pred_test)

print("\nFinal Test Metrics")
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)

Epoch 020 | Train Loss: 0.107478 | Test Loss: 0.156019
Epoch 040 | Train Loss: 0.072540 | Test Loss: 0.119087
Epoch 060 | Train Loss: 0.057399 | Test Loss: 0.107854
Epoch 080 | Train Loss: 0.040620 | Test Loss: 0.116496
Epoch 100 | Train Loss: 0.030271 | Test Loss: 0.119564

Final Test Metrics
RMSE: 2.6644288647347922
MAE : 1.6551471948623657
R2  : 0.8783086538314819
